# Payphone Full Import – Merge + GGUF on Colab T4

Merge your trained LoRA adapter and convert to GGUF entirely on Colab. Download a single **payphone-story.gguf** file. No heavy work on your machine.

**Before running:** Runtime → Change runtime type → **T4 GPU**  
**If OOM:** Runtime → Restart session, then run all from top (fresh GPU).

## 1. Install dependencies

In [ ]:
# Reduce CUDA fragmentation (run first; restart session if you had OOM)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q transformers peft bitsandbytes accelerate

## 2. Upload LoRA adapter

Upload `payphone-storyteller-lora.zip` (from Colab training download).

In [ ]:
from google.colab import files
import zipfile
import os

print("Upload payphone-storyteller-lora.zip")
uploaded = files.upload()
zip_path = list(uploaded.keys())[0] if uploaded else None
if not zip_path:
    raise FileNotFoundError("Upload payphone-storyteller-lora.zip")

os.makedirs("/content/lora", exist_ok=True)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/lora")
ADAPTER_PATH = "/content/lora/payphone-storyteller-lora"
if not os.path.exists(ADAPTER_PATH):
    ADAPTER_PATH = "/content/lora"
print(f"Adapter at: {ADAPTER_PATH}")

## 3. Merge LoRA into base model

In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE = "Qwen/Qwen2.5-7B-Instruct"
OUTPUT = "/content/merged_payphone"
os.makedirs(OUTPUT, exist_ok=True)

print("Loading base model (4-bit)...")
bnb = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)

print("Merging...")
model = model.merge_and_unload()

print("Saving...")
model.save_pretrained(OUTPUT)
tokenizer.save_pretrained(OUTPUT)
print("Done.")

## 4. Clone llama.cpp for GGUF conversion

In [ ]:
# Clone llama.cpp (script only; avoid pip deps that conflict with Colab)
!git clone -q --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
# Use PyPI gguf to avoid Colab package conflicts
!pip install -q gguf

## 5. Convert to GGUF (q8_0, ~7.5GB)

In [ ]:
import os
os.environ["NO_LOCAL_GGUF"] = "1"  # Use PyPI gguf, avoid dep conflicts
!cd /content/llama.cpp && python convert_hf_to_gguf.py /content/merged_payphone \
  --outfile /content/payphone-story.gguf \
  --outtype q8_0

## 6. Download GGUF

In [ ]:
from google.colab import files
import os
GGUF_PATH = "/content/payphone-story.gguf"
if os.path.exists(GGUF_PATH):
    files.download(GGUF_PATH)
    print("Download started. Place in project root, then: ollama create payphone-story -f Modelfile")
else:
    print("ERROR: GGUF not created. Check Cell 5 for errors. Merged model at: /content/merged_payphone")